<a href="https://colab.research.google.com/github/41Kasidet/PISA2022-Project/blob/main/2_Create_CSV_File.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine, text
import shutil

from google.colab import drive
drive.mount('/content/drive')
data_path = '/content/drive/MyDrive/Data Sci CU/Research with Kiatanan//'
# data_path = './'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
database_name = 'PISA_database.db'

conn = create_engine('sqlite:///'+data_path+database_name)

In [ ]:
import pandas as pd

# 1. สร้างคำสั่ง Index (รันครั้งแรกครั้งเดียว หรือใช้ IF NOT EXISTS)
# การทำ Index จะทำให้การ Join ข้อมูล 600,000 แถวเร็วขึ้นมหาศาล
index_queries = [
    "CREATE INDEX IF NOT EXISTS idx_id_t1 ON student_qqq_2022_1(student_qqq_id);",
    "CREATE INDEX IF NOT EXISTS idx_id_t2 ON student_qqq_2022_2(student_qqq_id);",
    "CREATE INDEX IF NOT EXISTS idx_id_t6 ON student_qqq_2022_6(student_qqq_id);",
    "CREATE INDEX IF NOT EXISTS idx_id_t7 ON student_qqq_2022_7(student_qqq_id);",
    "CREATE INDEX IF NOT EXISTS idx_id_t9 ON student_qqq_2022_9(student_qqq_id);"
]

# รันคำสั่งสร้าง Index ผ่าน connection
# Use conn.connect() to get a connection object, then execute on it.
with conn.connect() as connection:
    for q in index_queries:
        connection.execute(text(q))
    # Commit the changes (important for DDL statements in some databases)
    connection.commit()

# Dependence Variables

In [ ]:
# Recall Dependence Variable (Reading)

import pandas as pd

num_list = ['1','2','3','4','5','6','7','8','9','10']
pv_vars = [f'PV{i}MPRE' for i in num_list]

# Construct the CASE WHEN sum for missing values
sum_cases = []
for var in pv_vars:
    sum_cases.append(f"t9.{var}")

sum_expression = " + ".join(sum_cases)

# # Construct the WHERE clause to drop rows with any NULL in PV1MPRE to PV10MPRE
# where_conditions = []
# for var in pv_vars:
#     where_conditions.append(f"t9.{var} IS NOT NULL")
# where_clause = " AND ".join(where_conditions)

print(sum_expression)

query_data = f"""
      SELECT
          t1.student_qqq_id,
          t1.CNT as Country,
          ({sum_expression})/10 AS Score
      FROM student_qqq_2022_1 AS t1
      INNER JOIN student_qqq_2022_9 AS t9 ON t1.student_qqq_id = t9.student_qqq_id
      """

df_raw_scores_math = pd.read_sql(query_data, conn)

t9.PV1MPRE + t9.PV2MPRE + t9.PV3MPRE + t9.PV4MPRE + t9.PV5MPRE + t9.PV6MPRE + t9.PV7MPRE + t9.PV8MPRE + t9.PV9MPRE + t9.PV10MPRE


In [ ]:
df_raw_scores_math.head()

,student_qqq_id,Country,Score
0,1,ALB,249.9635
1,2,ALB,329.5964
2,3,ALB,338.3428
3,4,ALB,253.7627
4,5,ALB,481.7530


In [ ]:
# ~ 200,000 Math Score is NULL
df_raw_scores_math.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   student_qqq_id  613744 non-null  int64  
 1   Country         613744 non-null  object 
 2   Score           592123 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 14.0+ MB


In [ ]:
# Recall Dependence Variable (Reading)

import pandas as pd

num_list = ['1','2','3','4','5','6','7','8','9','10']
pv_vars = [f'PV{i}READ' for i in num_list]

# Construct the CASE WHEN sum for missing values
sum_cases = []
for var in pv_vars:
    sum_cases.append(f"t9.{var}")

sum_expression = " + ".join(sum_cases)

# # Construct the WHERE clause to drop rows with any NULL in PV1MPRE to PV10MPRE
# where_conditions = []
# for var in pv_vars:
#     where_conditions.append(f"t9.{var} IS NOT NULL")
# where_clause = " AND ".join(where_conditions)

print(sum_expression)

query_data = f"""
      SELECT
          t1.student_qqq_id,
          ({sum_expression})/10 AS Score
      FROM student_qqq_2022_1 AS t1
      INNER JOIN student_qqq_2022_9 AS t9 ON t1.student_qqq_id = t9.student_qqq_id
      """

df_raw_scores_reading = pd.read_sql(query_data, conn)

t9.PV1READ + t9.PV2READ + t9.PV3READ + t9.PV4READ + t9.PV5READ + t9.PV6READ + t9.PV7READ + t9.PV8READ + t9.PV9READ + t9.PV10READ


In [ ]:
df_raw_scores_reading.head()

,student_qqq_id,Score
0,1,249.8026
1,2,288.8999
2,3,311.7785
3,4,300.7753
4,5,486.6689


In [ ]:
df_raw_scores_reading.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   student_qqq_id  613744 non-null  int64  
 1   Score           613744 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 9.4 MB


In [ ]:
# Recall Dependence Variable (Science)

import pandas as pd

num_list = ['1','2','3','4','5','6','7','8','9','10']
pv_vars = [f'PV{i}SCIE' for i in num_list]

# Construct the CASE WHEN sum for missing values
sum_cases = []
for var in pv_vars:
    sum_cases.append(f"t9.{var}")

sum_expression = " + ".join(sum_cases)

# # Construct the WHERE clause to drop rows with any NULL in PV1MPRE to PV10MPRE
# where_conditions = []
# for var in pv_vars:
#     where_conditions.append(f"t9.{var} IS NOT NULL")
# where_clause = " AND ".join(where_conditions)

print(sum_expression)

query_data = f"""
      SELECT
          t1.student_qqq_id,
          ({sum_expression})/10 AS Score
      FROM student_qqq_2022_1 AS t1
      INNER JOIN student_qqq_2022_9 AS t9 ON t1.student_qqq_id = t9.student_qqq_id
      """

df_raw_scores_science = pd.read_sql(query_data, conn)

t9.PV1SCIE + t9.PV2SCIE + t9.PV3SCIE + t9.PV4SCIE + t9.PV5SCIE + t9.PV6SCIE + t9.PV7SCIE + t9.PV8SCIE + t9.PV9SCIE + t9.PV10SCIE


In [ ]:
df_raw_scores_science.head()

,student_qqq_id,Score
0,1,301.2603
1,2,303.5314
2,3,323.6492
3,4,210.1502
4,5,466.7572


In [ ]:
df_raw_scores_reading.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   student_qqq_id  613744 non-null  int64  
 1   Score           613744 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 9.4 MB


In [ ]:
# Rename 'Score' columns before merging to avoid conflicts
df_math_renamed = df_raw_scores_math.rename(columns={'Score': 'Math_Score'})[['student_qqq_id', 'Country', 'Math_Score']]
df_reading_renamed = df_raw_scores_reading.rename(columns={'Score': 'Reading_Score'})
df_science_renamed = df_raw_scores_science.rename(columns={'Score': 'Science_Score'})

# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_math_renamed, df_reading_renamed, on='student_qqq_id', how='inner')
df_merged_scores = pd.merge(df_merged_scores, df_science_renamed, on='student_qqq_id', how='inner')

# Display the first few rows of the merged DataFrame
display(df_merged_scores.head())
display(df_merged_scores.info())

,student_qqq_id,Country,Math_Score,Reading_Score,Science_Score
0,1,ALB,249.9635,249.8026,301.2603
1,2,ALB,329.5964,288.8999,303.5314
2,3,ALB,338.3428,311.7785,323.6492
3,4,ALB,253.7627,300.7753,210.1502
4,5,ALB,481.7530,486.6689,466.7572


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   student_qqq_id  613744 non-null  int64  
 1   Country         613744 non-null  object 
 2   Math_Score      592123 non-null  float64
 3   Reading_Score   613744 non-null  float64
 4   Science_Score   613744 non-null  float64
dtypes: float64(3), int64(1), object(1)
memory usage: 23.4+ MB


None

# Independence Variables

In [ ]:
query_summary = """
SELECT
    student_qqq_id,
    ST004D01T AS Gender,
    ST001D01T AS International_Grade,
    ST003D02T AS Birth_Month,
    ST003D03T AS Birth_Year
FROM student_qqq_2022_2
"""

df_student_character = pd.read_sql(query_summary, conn)

In [ ]:
df_student_character.head()

,student_qqq_id,Gender,International_Grade,Birth_Month,Birth_Year
0,1,1.0,10.0,5.0,2006.0
1,2,2.0,9.0,2.0,2006.0
2,3,2.0,9.0,8.0,2006.0
3,4,1.0,8.0,7.0,2006.0
4,5,1.0,10.0,1.0,2006.0


In [ ]:
df_student_character.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   student_qqq_id       613744 non-null  int64  
 1   Gender               613665 non-null  float64
 2   International_Grade  613744 non-null  float64
 3   Birth_Month          588336 non-null  float64
 4   Birth_Year           601061 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 23.4 MB


In [ ]:
# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_merged_scores, df_student_character, on='student_qqq_id', how='inner')

In [ ]:
df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   student_qqq_id       613744 non-null  int64  
 1   Country              613744 non-null  object 
 2   Math_Score           592123 non-null  float64
 3   Reading_Score        613744 non-null  float64
 4   Science_Score        613744 non-null  float64
 5   Gender               613665 non-null  float64
 6   International_Grade  613744 non-null  float64
 7   Birth_Month          588336 non-null  float64
 8   Birth_Year           601061 non-null  float64
dtypes: float64(7), int64(1), object(1)
memory usage: 42.1+ MB


In [ ]:
query_summary = """
SELECT
    student_qqq_id,
    CNTSCHID AS School_ID
FROM student_qqq_2022_1
"""

df_student_character = pd.read_sql(query_summary, conn)

In [ ]:
df_student_character.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   student_qqq_id  613744 non-null  int64  
 1   School_ID       613744 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 9.4 MB


In [ ]:
# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_merged_scores, df_student_character, on='student_qqq_id', how='inner')

In [ ]:
df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   student_qqq_id       613744 non-null  int64  
 1   Country              613744 non-null  object 
 2   Math_Score           592123 non-null  float64
 3   Reading_Score        613744 non-null  float64
 4   Science_Score        613744 non-null  float64
 5   Gender               613665 non-null  float64
 6   International_Grade  613744 non-null  float64
 7   Birth_Month          588336 non-null  float64
 8   Birth_Year           601061 non-null  float64
 9   School_ID            613744 non-null  float64
dtypes: float64(8), int64(1), object(1)
memory usage: 46.8+ MB


In [ ]:
query_summary = """
SELECT
    student_qqq_id,
    ESCS AS Socio_Econ_Status_Index
FROM student_qqq_2022_7
"""

df_student_socio_econ = pd.read_sql(query_summary, conn)

In [ ]:
df_student_socio_econ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 2 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   student_qqq_id           613744 non-null  int64  
 1   Socio_Econ_Status_Index  588276 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 9.4 MB


In [ ]:
# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_merged_scores, df_student_socio_econ, on='student_qqq_id', how='inner')

df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   student_qqq_id           613744 non-null  int64  
 1   Country                  613744 non-null  object 
 2   Math_Score               592123 non-null  float64
 3   Reading_Score            613744 non-null  float64
 4   Science_Score            613744 non-null  float64
 5   Gender                   613665 non-null  float64
 6   International_Grade      613744 non-null  float64
 7   Birth_Month              588336 non-null  float64
 8   Birth_Year               601061 non-null  float64
 9   School_ID                613744 non-null  float64
 10  Socio_Econ_Status_Index  588276 non-null  float64
dtypes: float64(9), int64(1), object(1)
memory usage: 51.5+ MB


In [ ]:
query_summary = """
SELECT
    student_qqq_id,
    ST250Q02JA AS Resource_Household_Computer,
    ST250Q03JA AS Resource_Household_Software,
    ST250Q04JA AS Resource_Household_Phone_Access_Internet,
    ST250Q05JA AS Resource_Household_WIFI
FROM student_qqq_2022_2
"""

df_student_hh_educ_resource = pd.read_sql(query_summary, conn)

In [ ]:
df_student_hh_educ_resource.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 5 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   student_qqq_id                            613744 non-null  int64  
 1   Resource_Household_Computer               583643 non-null  float64
 2   Resource_Household_Software               577205 non-null  float64
 3   Resource_Household_Phone_Access_Internet  585263 non-null  float64
 4   Resource_Household_WIFI                   582810 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 23.4 MB


In [ ]:
# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_merged_scores, df_student_hh_educ_resource, on='student_qqq_id', how='inner')

df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 15 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   student_qqq_id                            613744 non-null  int64  
 1   Country                                   613744 non-null  object 
 2   Math_Score                                592123 non-null  float64
 3   Reading_Score                             613744 non-null  float64
 4   Science_Score                             613744 non-null  float64
 5   Gender                                    613665 non-null  float64
 6   International_Grade                       613744 non-null  float64
 7   Birth_Month                               588336 non-null  float64
 8   Birth_Year                                601061 non-null  float64
 9   School_ID                                 613744 non-null  float64
 10  Socio_Econ_Status_In

In [ ]:
query_summary = """
SELECT
    student_qqq_id,
    OCOD1 AS Occupation_Mother,
    OCOD2 AS Occupation_Father
FROM student_qqq_2022_7
"""

df_student_parent_occupation = pd.read_sql(query_summary, conn)

df_student_parent_occupation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 3 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   student_qqq_id     613744 non-null  int64
 1   Occupation_Mother  613744 non-null  int64
 2   Occupation_Father  613744 non-null  int64
dtypes: int64(3)
memory usage: 14.0 MB


In [ ]:
# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_merged_scores, df_student_parent_occupation, on='student_qqq_id', how='inner')

df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 17 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   student_qqq_id                            613744 non-null  int64  
 1   Country                                   613744 non-null  object 
 2   Math_Score                                592123 non-null  float64
 3   Reading_Score                             613744 non-null  float64
 4   Science_Score                             613744 non-null  float64
 5   Gender                                    613665 non-null  float64
 6   International_Grade                       613744 non-null  float64
 7   Birth_Month                               588336 non-null  float64
 8   Birth_Year                                601061 non-null  float64
 9   School_ID                                 613744 non-null  float64
 10  Socio_Econ_Status_In

In [ ]:
query_summary = """
SELECT
    student_qqq_id,
    ST005Q01JA AS Education_Mother,
    ST007Q01JA AS Education_Father
FROM student_qqq_2022_2
"""

df_student_parent_education = pd.read_sql(query_summary, conn)

df_student_parent_education.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   student_qqq_id    613744 non-null  int64  
 1   Education_Mother  575718 non-null  float64
 2   Education_Father  564013 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 14.0 MB


In [ ]:
# Perform INNER JOIN operations
df_merged_scores = pd.merge(df_merged_scores, df_student_parent_education, on='student_qqq_id', how='inner')

df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 19 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   student_qqq_id                            613744 non-null  int64  
 1   Country                                   613744 non-null  object 
 2   Math_Score                                592123 non-null  float64
 3   Reading_Score                             613744 non-null  float64
 4   Science_Score                             613744 non-null  float64
 5   Gender                                    613665 non-null  float64
 6   International_Grade                       613744 non-null  float64
 7   Birth_Month                               588336 non-null  float64
 8   Birth_Year                                601061 non-null  float64
 9   School_ID                                 613744 non-null  float64
 10  Socio_Econ_Status_In

In [ ]:
query_summary = """
SELECT
    CNTSCHID AS School_ID,
    SCHLTYPE AS School_Type,
    SCHSIZE AS School_Size,
    STRATIO AS School_Student_Teacher_Ratio,
    SC001Q01TA AS School_Location,
    RATCMP1 AS School_Availability_Computer,
    RATCMP2 AS School_Availability_Computer_with_Internet,
    RATTAB AS School_Availability_Tablet,
    SC016Q01TA AS Funding_by_Gov,
    SC016Q02TA AS Funding_by_Fee_or_Charge,
    SC016Q03TA AS Funding_by_Donate,
    SC016Q04TA AS Funding_by_Others,
    SC175Q01JA AS Minute_Math_Class,
    STAFFSHORT AS Staff_Shortage,
    EDUSHORT AS Material_Shortage
FROM school_qqq_2022
"""

df_school = pd.read_sql(query_summary, conn)

df_school.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21629 entries, 0 to 21628
Data columns (total 15 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   School_ID                                   21629 non-null  int64  
 1   School_Type                                 20007 non-null  float64
 2   School_Size                                 18643 non-null  float64
 3   School_Student_Ratio                        17884 non-null  float64
 4   School_Location                             20200 non-null  float64
 5   School_Availability_Computer                17472 non-null  float64
 6   School_Availability_Computer_with_Internet  17567 non-null  float64
 7   School_Availability_Tablet                  18242 non-null  float64
 8   Funding_by_Gov                              18751 non-null  float64
 9   Funding_by_Fee_or_Charge                    18409 non-null  float64
 10  Funding_by

In [ ]:
missing_by_column = df_merged_scores.isnull().sum()

print(missing_by_column)

student_qqq_id                                  0
Country                                         0
Math_Score                                  21621
Reading_Score                                   0
Science_Score                                   0
Gender                                         79
International_Grade                             0
Birth_Month                                 25408
Birth_Year                                  12683
School_ID                                       0
Socio_Econ_Status_Index                     25468
Resource_Household_Computer                 30101
Resource_Household_Software                 36539
Resource_Household_Phone_Access_Internet    28481
Resource_Household_WIFI                     30934
Occupation_Mother                               0
Occupation_Father                               0
Education_Mother                            38026
Education_Father                            49731
dtype: int64


In [ ]:
df_merged_scores = pd.merge(df_merged_scores, df_school, on='School_ID', how='inner')

df_merged_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 33 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   student_qqq_id                              613744 non-null  int64  
 1   Country                                     613744 non-null  object 
 2   Math_Score                                  592123 non-null  float64
 3   Reading_Score                               613744 non-null  float64
 4   Science_Score                               613744 non-null  float64
 5   Gender                                      613665 non-null  float64
 6   International_Grade                         613744 non-null  float64
 7   Birth_Month                                 588336 non-null  float64
 8   Birth_Year                                  601061 non-null  float64
 9   School_ID                                   613744 non-null  float64
 

# Data Manipulation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
data_path = '/content/drive/MyDrive/Data Sci CU/Research with Kiatanan//'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# Assuming 'your_file_name.csv' is the name of your CSV file.
# Please replace 'your_file_name.csv' with the actual name of your CSV file.
csv_file_name = 'PISA_2022_Dataset_Selected_Columns.csv'
df = pd.read_csv(data_path + csv_file_name)

display(df.head())

,student_qqq_id,Country,Math_Score,Reading_Score,Science_Score,Gender,International_Grade,Birth_Month,Birth_Year,School_ID,...,School_Availability_Computer,School_Availability_Computer_with_Internet,School_Availability_Tablet,Funding_by_Gov,Funding_by_Fee_or_Charge,Funding_by_Donate,Funding_by_Others,Minute_Math_Class,Staff_Shortage,Material_Shortage
0,1,ALB,249.9635,249.8026,301.2603,1.0,10.0,5.0,2006.0,800282.0,...,0.2813,0.5556,0.6250,90.0,10.0,0.0,0.0,45.0,-0.0097,-0.2805
1,2,ALB,329.5964,288.8999,303.5314,2.0,9.0,2.0,2006.0,800115.0,...,0.3205,0.0000,0.0000,100.0,0.0,0.0,0.0,45.0,-1.4551,2.9595
2,3,ALB,338.3428,311.7785,323.6492,2.0,9.0,8.0,2006.0,800242.0,...,0.4865,NaN,0.0000,50.0,50.0,0.0,0.0,45.0,0.1683,0.1753
3,4,ALB,253.7627,300.7753,210.1502,1.0,8.0,7.0,2006.0,800245.0,...,0.0000,0.0000,0.0000,100.0,0.0,0.0,0.0,45.0,-1.4551,0.4399
4,5,ALB,481.7530,486.6689,466.7572,1.0,10.0,1.0,2006.0,800285.0,...,0.0441,1.0000,0.1101,80.0,10.0,10.0,0.0,45.0,-0.7828,0.1000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 33 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   student_qqq_id                              613744 non-null  int64  
 1   Country                                     613744 non-null  object 
 2   Math_Score                                  592123 non-null  float64
 3   Reading_Score                               613744 non-null  float64
 4   Science_Score                               613744 non-null  float64
 5   Gender                                      613665 non-null  float64
 6   International_Grade                         613744 non-null  float64
 7   Birth_Month                                 588336 non-null  float64
 8   Birth_Year                                  601061 non-null  float64
 9   School_ID                                   613744 non-null  float64
 

## Modify School_Location to Urban/Rural


In [ ]:
df['School_Location'].value_counts(dropna = False)

,count
School_Location,
4.0,158823
3.0,158362
2.0,114082
5.0,81331
1.0,57253
NaN,35932
6.0,7961


In [ ]:
import numpy as np

conditions = [
    df['School_Location'] == 1,
    df['School_Location'].isin([2, 3, 4, 5, 6])
]
choices = [1, 2]

df['School_Location_Urban/Rural_1'] = np.select(conditions, choices, default=np.nan)

In [ ]:
print(df['School_Location_Urban/Rural_1'].value_counts(dropna=False))

School_Location_Urban/Rural_1
2.0    520559
1.0     57253
NaN     35932
Name: count, dtype: int64


In [ ]:
import numpy as np

conditions = [
    df['School_Location'].isin([1,2]),
    df['School_Location'].isin([3, 4, 5, 6])
]
choices = [1, 2]

df['School_Location_Urban/Rural_2'] = np.select(conditions, choices, default=np.nan)

print(df['School_Location_Urban/Rural_2'].value_counts(dropna=False))

School_Location_Urban/Rural_2
2.0    406477
1.0    171335
NaN     35932
Name: count, dtype: int64


## Modify Parent Occupation

In [ ]:
import numpy as np

# Father
df['Occupation_Mother_Category'] = df['Occupation_Mother'] // 1000
df['Occupation_Father_Category'] = df['Occupation_Father'] // 1000

def job_group(number):
  if number in [1, 2, 3]:
    return 1
  elif number in [4, 5, 6, 7, 8]:
    return 2
  elif number == 9:
    return 9
  elif number == 0:
    return 0
  else:
    return np.nan

df['Occupation_Mother_Category'] = df['Occupation_Mother_Category'].apply(job_group)
df['Occupation_Father_Category'] = df['Occupation_Father_Category'].apply(job_group)


In [ ]:
print(df['Occupation_Mother_Category'].value_counts(dropna=False))
print(df['Occupation_Father_Category'].value_counts(dropna=False))

Occupation_Mother_Category
9    239964
1    170055
2    134431
0     69294
Name: count, dtype: int64
Occupation_Father_Category
2    191663
9    182105
1    146535
0     93441
Name: count, dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 37 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   student_qqq_id                              613744 non-null  int64  
 1   Country                                     613744 non-null  object 
 2   Math_Score                                  592123 non-null  float64
 3   Reading_Score                               613744 non-null  float64
 4   Science_Score                               613744 non-null  float64
 5   Gender                                      613665 non-null  float64
 6   International_Grade                         613744 non-null  float64
 7   Birth_Month                                 588336 non-null  float64
 8   Birth_Year                                  601061 non-null  float64
 9   School_ID                                   613744 non-null  float64
 

In [ ]:
file_name = 'PISA_2022_Dataset_Selected_Columns.csv'

# df_merged_scores.to_csv(data_path + file_name, index=False)
df.to_csv(data_path + file_name, index=False)
print("บันทึกเสร็จสิ้น")

บันทึกเสร็จสิ้น
